In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip install faiss-cpu
!pip install rank-bm25

In [3]:
%cd /content/gdrive/MyDrive/SPTAR_ADV/SPTAR

/content/gdrive/MyDrive/SPTAR_ADV/SPTAR


In [4]:
import pandas as pd
import numpy as np
import datetime
import os
import re
from sentence_transformers import SentenceTransformer
import faiss
import json
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
import torch
import time, hashlib, tempfile, shutil
from typing import Dict, Any, List, Optional

## Document to Index

In [5]:
train_path = pd.read_csv('soft_prompt/data/law/prompt_tuning_train_text.csv')
generated_train_path = pd.read_csv('inference_output/law/weak_queries_50_exaone-7b_523_prompt_3.csv', sep="\t")
test_path = pd.read_csv('soft_prompt/data/law/prompt_tuning_test_text.csv')

non_labeled_corpus = []
with open('retrieve/datasets/raw/beir/law/corpus_filtered.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        non_labeled_corpus.append(json.loads(line))

generated_query = []
with open('inference_output/law/weak_queries_50_exaone-7b_523_prompt_3.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        generated_query.append(json.loads(line))

In [6]:
real_query_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(generated_query)):
        if str(generated_train_path['query-id'][i]) == str(generated_query[j]['_id']):
            real_query_text.append(generated_query[j]['text'])

generated_train_path['text_x'] = real_query_text

real_corpus_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(non_labeled_corpus)):
        if str(generated_train_path['corpus-id'][i]) == str(non_labeled_corpus[j]['_id']):
            real_corpus_text.append(non_labeled_corpus[j]['text'])

generated_train_path['text_y'] = real_corpus_text

generated_train_path.head(2)

,query-id,corpus-id,score,text_x,text_y
0,5000001,2613,1,하자와의 분쟁에 대한 조정이 이루어지지 않아 강제경매를 진행한 경우 그 부동산을 취...,가등기담보 등에 관한 법률 15조 제15조(담보가등기권리의 소멸) 담보가등기를 마친...
1,5000002,2614,1,서울에서 자영업자로 음식점을 운영하고 있습니다 저녁시간에는 손님이 별로 없어 종업원...,가사근로자의 고용개선 등에 관한 법률 11조 제3장 가사서비스의 제공 제11조(가사...


In [7]:
new_train_path = pd.concat([generated_train_path, train_path])
new_train_path.reset_index(drop=True)
all_corpus = pd.concat([new_train_path, test_path]).drop_duplicates().reset_index(drop=True)
all_corpus.shape

(3643, 5)

In [8]:
class PrebuiltIndexNotFound(Exception): ...
class ArtifactMissing(Exception): ...

# util
def _safe_makedirs(p: str): os.makedirs(p, exist_ok=True)

def _atomic_write_bytes(dst: str, data: bytes):
    d = os.path.dirname(dst); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, dst)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _save_json(path: str, obj: Any):
    _atomic_write_bytes(path, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))

def _load_json(path: str) -> Any:
    if not os.path.exists(path): raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def _save_numpy_atomic(path: str, arr: np.ndarray):
    d = os.path.dirname(path); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            np.save(f, arr); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _load_numpy(path: str) -> np.ndarray:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return np.load(path)

def _save_faiss(path: str, index: faiss.Index):
    _safe_makedirs(os.path.dirname(path)); faiss.write_index(index, path)

def _load_faiss(path: str) -> faiss.Index:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return faiss.read_index(path)

# 코퍼스 키: 문서+instruction만 반영
def _hash_corpus(docs: List[str], instruction: bool) -> str:
    h = hashlib.sha256()
    h.update(b"inst1" if instruction else b"inst0")
    for d in docs:
        h.update(b"\x1e"); h.update(d.encode("utf-8"))
    return h.hexdigest()[:16]

def _paths_corpus(store_dir: str, corpus_key: str) -> Dict[str, str]:
    root = os.path.join(store_dir, corpus_key)
    return {
        "root": root,
        "corpus": os.path.join(root, "corpus.json"),
        "corpus_config": os.path.join(root, "corpus_config.json"),
        "bm25_tokens": os.path.join(root, "bm25_tokens.json"),
        "variants_dir": os.path.join(root, "variants"),
    }

def _paths_variant(store_dir: str, corpus_key: str, variant: str) -> Dict[str, str]:
    vd = os.path.join(store_dir, corpus_key, "variants", variant)
    return {
        "dir": vd,
        "config": os.path.join(vd, "config.json"),
        "dense_npy": os.path.join(vd, "dense_emb.npy"),
        "faiss_index": os.path.join(vd, "faiss.index"),
    }

# 임베딩 저장
def build_and_save_index(
    *,
    docs: List[str],
    instruction: bool,
    variant: str,                 # ex) "bge-m3" | "sbert" | "other"
    embed_model_name: str,        # ex) "BAAI/bge-m3", "sentence-transformers/all-MiniLM-L6-v2", .
    method: str,                  # "dense" | "faiss" | "bm25"
    store_dir: str = "embedded_docs",
    corpus_key: Optional[str] = None,   # 미지정 시 자동 생성
    overwrite_variant: bool = False,    # 같은 variant 덮어쓸지
    build_bm25_once: bool = False,      # BM25 토큰도 함께 만들고 싶으면 True
) -> str:
    # instruction 프리픽스 적용
    _docs = [f"Document: {d}" for d in docs] if instruction else list(docs)
    corpus_key = corpus_key or _hash_corpus(_docs, instruction)

    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)

    # 코퍼스 메타/문서 저장(없으면)
    _safe_makedirs(PC["root"])
    if not os.path.exists(PC["corpus"]):
        _save_json(PC["corpus"], _docs)
        _save_json(PC["corpus_config"], {
            "instruction": instruction,
            "created_at": int(time.time()),
            "version": 1
        })

    # BM25 토큰 생성(옵션, 중복 방지)
    if build_bm25_once and not os.path.exists(PC["bm25_tokens"]):
        tok = AutoTokenizer.from_pretrained(embed_model_name)
        tokenized = [tok.tokenize(d) for d in _docs]
        _save_json(PC["bm25_tokens"], tokenized)

    # 변종 생성
    if (os.path.exists(PV["config"]) or os.path.exists(PV["dense_npy"]) or os.path.exists(PV["faiss_index"])) and not overwrite_variant:
        return corpus_key  # 이미 있음

    _safe_makedirs(PV["dir"])

    if method == "bm25":
        # 변종에 별도 파일은 없음(코퍼스 공용 bm25_tokens 사용)
        _save_json(PV["config"], {
            "variant": variant, "embed_model_name": embed_model_name,
            "method": "bm25", "created_at": int(time.time())
        })
        return corpus_key

    if method not in {"dense", "faiss"}:
        raise ValueError("method must be one of {'bm25','dense','faiss'}")

    model = SentenceTransformer(embed_model_name)
    emb = model.encode(_docs, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _save_numpy_atomic(PV["dense_npy"], emb)

    if method == "faiss":
        dim = emb.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(emb)
        _save_faiss(PV["faiss_index"], index)

    _save_json(PV["config"], {
        "variant": variant,
        "embed_model_name": embed_model_name,
        "method": method,
        "dim": int(emb.shape[1]) if method in {"dense","faiss"} else None,
        "created_at": int(time.time())
    })
    return corpus_key

# 로드
def load_index(
    *,
    store_dir: str,
    corpus_key: str,
    variant: str,
    use_gpu_for_faiss: bool = True
) -> Dict[str, Any]:
    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)
    if not os.path.exists(PC["corpus"]):
        raise PrebuiltIndexNotFound(f"Corpus not found: {PC['root']}")

    docs = _load_json(PC["corpus"])
    if not os.path.exists(PV["config"]):
        raise PrebuiltIndexNotFound(f"Variant not found: {PV['dir']}")

    vcfg = _load_json(PV["config"])
    method = vcfg["method"]

    if method == "bm25":
        if not os.path.exists(PC["bm25_tokens"]):
            raise ArtifactMissing("bm25_tokens.json missing for corpus")
        tokens = _load_json(PC["bm25_tokens"])
        tokenizer = AutoTokenizer.from_pretrained(vcfg["embed_model_name"])
        bm25 = BM25Okapi(tokens)
        return {"method": "bm25", "docs": docs, "bm25": bm25, "tokenizer": tokenizer}

    if method == "dense":
        if not os.path.exists(PV["dense_npy"]):
            raise ArtifactMissing("dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "dense", "docs": docs, "docs_embeddings": emb, "dense_model": model}

    if method == "faiss":
        if not (os.path.exists(PV["dense_npy"]) and os.path.exists(PV["faiss_index"])):
            raise ArtifactMissing("faiss.index or dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        index = _load_faiss(PV["faiss_index"])  # CPU
        if use_gpu_for_faiss and torch.cuda.is_available():
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "faiss", "docs": docs, "docs_embeddings": emb, "dense_model": model, "faiss_index": index}

    raise ValueError(f"unknown method in variant config: {method}")

# 검색
def search(query: str, k: int, handle: Dict[str, Any]):
    m = handle["method"]; docs = handle["docs"]
    if m == "bm25":
        tok = handle["tokenizer"]; bm25 = handle["bm25"]
        q_tokens = tok.tokenize(query)
        scores = bm25.get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        return [(docs[i], float(scores[i])) for i in idx]
    if m == "dense":
        model = handle["dense_model"]; emb = handle["docs_embeddings"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")
        sims = emb @ qv
        idx = np.argsort(sims)[::-1][:k]
        return [(docs[i], float(sims[i])) for i in idx]
    if m == "faiss":
        model = handle["dense_model"]; index = handle["faiss_index"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        D, I = index.search(qv, k)
        return [(docs[int(i)], float(D[0, j])) for j, i in enumerate(I[0])]
    raise ValueError("unknown method in handle")


In [9]:
variants      = ["bge-m3", "ko-legal-sbert", "qwen3-embedding"]
model_list    = ["upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"]#, "Day1Kim/Qwen3-Embedding-0.6B-Korean"]
method_list   = ["bm25", "dense", "faiss"]
corpus_key    = "corpus-all"
# corpus_key = "law-v1"

docs = list(all_corpus['text_y'])

In [ ]:

# base_variant = "bge-m3"
# for method in method_list:
#     # (모델 × 메서드) 조합으로 variant 분기
#     variant_name = f"{base_variant}@{method}"

#     build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
#     build_and_save_index(
#         docs=docs,
#         instruction=True,
#         variant=variant_name,
#         embed_model_name="upskyy/bge-m3-korean",
#         method=method,                  # "bm25" | "dense" | "faiss"
#         store_dir="embedded_docs",
#         corpus_key=corpus_key,
#         overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
#         build_bm25_once=build_bm25
#     )


In [ ]:

# base_variant = "ko-legal-sbert"
# for method in method_list:
#     # (모델 × 메서드) 조합으로 variant 분기
#     variant_name = f"{base_variant}@{method}"

#     build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
#     build_and_save_index(
#         docs=docs,
#         instruction=False,
#         variant=variant_name,
#         embed_model_name= "woong0322/ko-legal-sbert-finetuned",
#         method=method,                  # "bm25" | "dense" | "faiss"
#         store_dir="embedded_docs",
#         corpus_key=corpus_key,
#         overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
#         build_bm25_once=build_bm25
#     )


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
# OOM 문제로 로드 불가능

# base_variant = "qwen3-embedding"
# for method in method_list:

#     # (모델 × 메서드) 조합으로 variant 분기
#     variant_name = f"{base_variant}@{method}"

#     build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
#     build_and_save_index(
#         docs=docs,
#         instruction=False,
#         variant=variant_name,
#         embed_model_name="Day1Kim/Qwen3-Embedding-0.6B-Korean",
#         method=method,                  # "bm25" | "dense" | "faiss"
#         store_dir="embedded_docs",
#         corpus_key=corpus_key,
#         overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
#         build_bm25_once=build_bm25
#     )


## Train Retriever

In [ ]:
'''
실험 목적
### 생성된 weak query가 실제로 연관된 문서를 잘 검색하는가?를 알기 위함
### 생성 전(train_path)의 데이터로부터 검색기를 학습시킨다
### 생성된 weak query data(train_path + generated_train_path) 로부터 검색기를 학습시킨다

### 둘의 결과를 평가한다. (by test_path)
### 학습 평가는 정확도로 총 2개를 비교한다. top=1으로 golden docs를 가져왔는지 & top=15 중 golden docs가 포함되어있는지

실험 세팅
- embedding model : "upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"
- method : "bm25", "dense", "faiss"
- corpus_key : "corpus-all"
  -> 해당 파일 안에 : bge@bm25, bge@dense, bge@faiss, sbert@bm25, sbert@dense, sbert@faiss 등의 파일이 존재함 -> 이 6개의 조합에 대한 실험 진행
- corpus for train : train_path['text_y'] 와 train_path['text_y'] + generated_train_path['text_y']로 비교

'''

In [9]:
# -*- coding: utf-8 -*-
from typing import List, Tuple, Dict, Optional
import re
import pandas as pd


def _normalize(t: str) -> str:
    # 공백/개행 제거 + 일부 기호 정규화 (필요시 규칙 추가)
    t = t.replace('\u00B7','·').replace('ㆍ','·')
    t = re.sub(r'\s+', '', t)
    t = re.sub(r'[“”„‟″]', '"', t)
    t = re.sub(r"[’‘′`']", "'", t)
    t = t.replace('–','-').replace('—','-')
    return t

def head_key(t: str, n: int = 100) -> str:
    return _normalize(t)[-n:]

def build_tail100_index(
    corpus_texts: List[str],
    corpus_ids: List[int],
    n: int = 100
) -> Dict[str, List[int]]:
    assert len(corpus_texts) == len(corpus_ids)
    idx: Dict[str, List[int]] = {}
    for cid, txt in zip(corpus_ids, corpus_texts):
        k = head_key(txt, n=n)
        idx.setdefault(k, []).append(int(cid))
    return idx


def map_text_to_corpus_id(
    text: str,
    index: Dict[str, List[int]],
    n: int = 100,
    on_collision: str = "first",  # {"first","none","error"}
) -> Optional[int]:
    k = head_key(text, n=n)
    hit = index.get(k)
    if not hit:
        return None
    if len(hit) == 1:
        return hit[0]
    if on_collision == "first":
        return hit[0]
    elif on_collision == "none":
        return None
    elif on_collision == "error":
        raise ValueError(f"Ambiguous head-{n} match: {hit}")
    else:
        raise ValueError("on_collision must be one of {'first','none','error'}")

def attach_corpus_ids_to_results(
    results: List[Tuple[str, float]],
    index: Dict[str, List[int]],
    n: int = 100,
    on_collision: str = "first"
) -> List[Tuple[Optional[int], str, float]]:
    """
    returns: [(corpus_id_or_None, text, score), ...]
    """
    out = []
    for text, score in results:
        cid = map_text_to_corpus_id(text, index, n=n, on_collision=on_collision)
        out.append((cid, text, score))
    return out

In [10]:
import math
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
import os
os.environ["WANDB_DISABLED"] = "true"

def train_and_build_index(model_name, variant, train_pairs, dev_pairs, batch_size, num_epochs, docs, method, corpus_key):
    os.makedirs('train_retriever_result', exist_ok=True)
    output_path = f"train_retriever_result/{variant}-contrastive-{corpus_key}"

    model = SentenceTransformer(model_name)

    # 학습 데이터
    train_data = [InputExample(texts=[a, b], label=int(y)) for a, b, y in train_pairs]
    train_dataloader = DataLoader(train_data, shuffle=True, batch_size=batch_size)

    train_loss = losses.ContrastiveLoss(model, margin=0.5)

    # 검증 데이터
    dev_sents1 = [a for a, _, _ in dev_pairs]
    dev_sents2 = [b for _, b, _ in dev_pairs]
    dev_labels = [int(y) for _, _, y in dev_pairs]
    bin_eval = evaluation.BinaryClassificationEvaluator(dev_sents1, dev_sents2, dev_labels, name="dev-bin")

    warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

    # --- Early Stopping 설정 ---
    patience = 4                 # 개선 없으면 중단할 최대 횟수
    best_score = -1              # 최고 점수 저장
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        # 한 epoch 학습
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            evaluator=bin_eval,        # 여기서 dev 평가 수행
            epochs=1,                  # 매번 1 epoch만 돌림
            warmup_steps=warmup_steps,
            evaluation_steps=200,
            output_path=output_path,
            save_best_model=True
        )


        # 성능 측정 (여기서는 accuracy 기준)
        results = bin_eval(model)
        print(list(results.keys()))
        score = results['dev-bin_cosine_accuracy']   # dict에서 accuracy만 꺼내서 비교

        if score > best_score:
            best_score = score
            patience_counter = 0
            print(f"✅ 성능 향상: {score:.4f} (best 갱신)")
        else:
            patience_counter += 1
            print(f"⚠️ 성능 개선 없음 (연속 {patience_counter}회)")

        if patience_counter >= patience:
            print(f"⛔ Early stopping 발동 (patience={patience}) at epoch {epoch+1}")
            break
    # 학습된 모델로 인덱스 빌드
    variant_name=f"{variant}@{method}"
    build_bm25 = (method == "bm25")
    instruction = (model_name=="upskyy/bge-m3-korean")

    build_and_save_index(
        docs=docs,
        instruction=instruction,
        variant=variant_name,
        embed_model_name=output_path,
        method=method,
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,
        build_bm25_once=build_bm25
    )


In [11]:
train_pairs = []
dev_pairs = []
generated_train_pairs = []

for i in range(train_path.shape[0]):
  train_pairs.append((train_path['text_x'].iloc[i], train_path['text_y'].iloc[i], train_path['score'].iloc[i]))

for i in range(test_path.shape[0]):
  dev_pairs.append((test_path['text_x'].iloc[i], test_path['text_y'].iloc[i], test_path['score'].iloc[i]))


for i in range(new_train_path.shape[0]):
  generated_train_pairs.append((new_train_path['text_x'].iloc[i], new_train_path['text_y'].iloc[i], new_train_path['score'].iloc[i]))

In [12]:
variants      = ["bge-m3", "ko-legal-sbert"]
model_list    = ["upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"]
method_list   = ["bm25", "dense", "faiss"]
corpus_key    = "corpus-train"
corpus_key    = "corpus-generated-train"
docs = list(all_corpus['text_y'])

In [14]:
 train_and_build_index(
     model_name="upskyy/bge-m3-korean",
     variant="bge-m3",
     train_pairs=train_pairs,
     dev_pairs=dev_pairs,
     batch_size=1,
     num_epochs=50,
     docs=docs,
     method='bm25',
     corpus_key='corpus-train-50')

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch 1/50


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Dev-bin Cosine Accuracy,Dev-bin Cosine Accuracy Threshold,Dev-bin Cosine F1,Dev-bin Cosine F1 Threshold,Dev-bin Cosine Precision,Dev-bin Cosine Recall,Dev-bin Cosine Ap,Dev-bin Cosine Mcc
200,No log,No log,0.883365,0.508581,0.814159,0.481102,0.816568,0.811765,0.895549,0.725058
400,No log,No log,0.883365,0.601222,0.828169,0.590262,0.794595,0.864706,0.898451,0.741645
600,0.028400,No log,0.910134,0.651304,0.860465,0.633270,0.850575,0.870588,0.925442,0.792222
800,0.028400,No log,0.921606,0.668017,0.878338,0.665492,0.886228,0.870588,0.933062,0.820588
1000,0.015700,No log,0.927342,0.690360,0.888889,0.690360,0.883721,0.894118,0.930993,0.834946
1200,0.015700,No log,0.934990,0.676508,0.902857,0.660432,0.877778,0.929412,0.939943,0.854854
1400,0.015700,No log,0.942639,0.629382,0.917127,0.629382,0.864583,0.976471,0.947738,0.877297
1600,0.010500,No log,0.944551,0.689522,0.915942,0.689522,0.902857,0.929412,0.941671,0.874790
1800,0.010500,No log,0.934990,0.740997,0.901734,0.732304,0.886364,0.917647,0.935960,0.853471
2000,0.007100,No log,0.915870,0.710861,0.869048,0.710861,0.879518,0.858824,0.905018,0.807213


...

Step,Training Loss,Validation Loss,Dev-bin Cosine Accuracy,Dev-bin Cosine Accuracy Threshold,Dev-bin Cosine F1,Dev-bin Cosine F1 Threshold,Dev-bin Cosine Precision,Dev-bin Cosine Recall,Dev-bin Cosine Ap,Dev-bin Cosine Mcc
200,No log,No log,0.948375,0.668875,0.922190,0.668875,0.903955,0.941176,0.923101,0.883990
400,No log,No log,0.948375,0.672260,0.922190,0.672260,0.903955,0.941176,0.927367,0.883990
600,0.002500,No log,0.950287,0.629448,0.925714,0.623599,0.900000,0.952941,0.922973,0.889223
800,0.002500,No log,0.950287,0.570708,0.927778,0.558210,0.878947,0.982353,0.920062,0.893255
1000,0.000300,No log,0.950287,0.590268,0.927778,0.577048,0.878947,0.982353,0.921005,0.893255
1200,0.000300,No log,0.948375,0.622558,0.924791,0.601086,0.878307,0.976471,0.918970,0.888538
1400,0.000300,No log,0.942639,0.584642,0.916201,0.569429,0.872340,0.964706,0.914673,0.875318
1600,0.000200,No log,0.944551,0.623029,0.918768,0.609585,0.877005,0.964706,0.912149,0.879117
1800,0.000200,No log,0.938815,0.674231,0.910112,0.674231,0.870968,0.952941,0.903848,0.865885
2000,0.000200,No log,0.934990,0.595519,0.907104,0.595519,0.846939,0.976471,0.902085,0.862629


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


['dev-bin_cosine_accuracy', 'dev-bin_cosine_accuracy_threshold', 'dev-bin_cosine_f1', 'dev-bin_cosine_f1_threshold', 'dev-bin_cosine_precision', 'dev-bin_cosine_recall', 'dev-bin_cosine_ap', 'dev-bin_cosine_mcc']
⚠️ 성능 개선 없음 (연속 3회)
Epoch 8/50


Step,Training Loss,Validation Loss,Dev-bin Cosine Accuracy,Dev-bin Cosine Accuracy Threshold,Dev-bin Cosine F1,Dev-bin Cosine F1 Threshold,Dev-bin Cosine Precision,Dev-bin Cosine Recall,Dev-bin Cosine Ap,Dev-bin Cosine Mcc
200,No log,No log,0.940727,0.673664,0.913165,0.673664,0.871658,0.958824,0.942327,0.870600
400,No log,No log,0.948375,0.636620,0.925208,0.624945,0.874346,0.982353,0.942873,0.889499
600,0.002600,No log,0.954111,0.611295,0.932961,0.611295,0.888298,0.982353,0.942953,0.900840
800,0.002600,No log,0.950287,0.642319,0.925714,0.642319,0.900000,0.952941,0.943534,0.889223
1000,0.000300,No log,0.948375,0.697605,0.923513,0.640929,0.890710,0.958824,0.938277,0.885999
1200,0.000300,No log,0.948375,0.677679,0.923513,0.640828,0.890710,0.958824,0.940449,0.885999
1400,0.000300,No log,0.948375,0.612922,0.923077,0.612922,0.895028,0.952941,0.940227,0.885270
1600,0.000100,No log,0.946463,0.711271,0.920455,0.654536,0.890110,0.952941,0.942482,0.881342
1800,0.000100,No log,0.950287,0.716442,0.925714,0.716442,0.900000,0.952941,0.947867,0.889223
2000,0.000300,No log,0.940727,0.702293,0.912181,0.658719,0.879781,0.947059,0.948034,0.868881


['dev-bin_cosine_accuracy', 'dev-bin_cosine_accuracy_threshold', 'dev-bin_cosine_f1', 'dev-bin_cosine_f1_threshold', 'dev-bin_cosine_precision', 'dev-bin_cosine_recall', 'dev-bin_cosine_ap', 'dev-bin_cosine_mcc']
⚠️ 성능 개선 없음 (연속 4회)
⛔ Early stopping 발동 (patience=4) at epoch 8


In [13]:
 train_and_build_index(
     model_name="upskyy/bge-m3-korean",
     variant="bge-m3",
     train_pairs=generated_train_pairs,
     dev_pairs=dev_pairs,
     batch_size=1,
     num_epochs=50,
     docs=docs,
     method='bm25',
     corpus_key='corpus-generated-train-50')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

...

['dev-bin_cosine_accuracy', 'dev-bin_cosine_accuracy_threshold', 'dev-bin_cosine_f1', 'dev-bin_cosine_f1_threshold', 'dev-bin_cosine_precision', 'dev-bin_cosine_recall', 'dev-bin_cosine_ap', 'dev-bin_cosine_mcc']
⚠️ 성능 개선 없음 (연속 3회)
Epoch 9/50


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss,Validation Loss,Dev-bin Cosine Accuracy,Dev-bin Cosine Accuracy Threshold,Dev-bin Cosine F1,Dev-bin Cosine F1 Threshold,Dev-bin Cosine Precision,Dev-bin Cosine Recall,Dev-bin Cosine Ap,Dev-bin Cosine Mcc
200,No log,No log,0.915870,0.771071,0.868805,0.621412,0.861272,0.876471,0.917550,0.804869
400,No log,No log,0.917782,0.740802,0.874636,0.635868,0.867052,0.882353,0.920097,0.813545
600,0.004600,No log,0.917782,0.762436,0.871345,0.621813,0.866279,0.876471,0.919932,0.808879
800,0.004600,No log,0.919694,0.760054,0.872404,0.660798,0.880240,0.864706,0.927062,0.811832
1000,0.000900,No log,0.923518,0.737850,0.877301,0.737850,0.916667,0.841176,0.928939,0.823495
1200,0.000900,No log,0.921606,0.759297,0.878613,0.616366,0.863636,0.894118,0.925640,0.818915
1400,0.000900,No log,0.921606,0.781616,0.875380,0.703900,0.905660,0.847059,0.925093,0.819265
1600,0.000200,No log,0.921606,0.787814,0.877612,0.665039,0.890909,0.864706,0.924877,0.820163
1800,0.000200,No log,0.921606,0.771692,0.875380,0.732573,0.905660,0.847059,0.927659,0.819265
2000,0.000100,No log,0.921606,0.792462,0.877612,0.708978,0.890909,0.864706,0.925785,0.820163


['dev-bin_cosine_accuracy', 'dev-bin_cosine_accuracy_threshold', 'dev-bin_cosine_f1', 'dev-bin_cosine_f1_threshold', 'dev-bin_cosine_precision', 'dev-bin_cosine_recall', 'dev-bin_cosine_ap', 'dev-bin_cosine_mcc']
⚠️ 성능 개선 없음 (연속 4회)
⛔ Early stopping 발동 (patience=4) at epoch 9


Token indices sequence length is longer than the specified maximum sequence length for this model (1057 > 512). Running this sequence through the model will result in indexing errors


In [2]:
def search_inference(query, corpus_key="corpus-all", variant="bge-m3@faiss", use_gpu_for_faiss=False):
    index_setting = load_index(store_dir="embedded_docs", corpus_key=corpus_key, variant=variant, use_gpu_for_faiss=use_gpu_for_faiss)
    if 'bge' in variant:
      query = f"Query: {query}"
    res = search(query, k=15, handle=index_setting)
    return res


def evaluate(all_corpus, test_path, corpus_key="corpus-all", variant="bge-m3@faiss", use_gpu_for_faiss=False):
    docs = list(all_corpus['text_y'])
    corpus_ids = list(all_corpus['corpus-id'])

    correct15, total15 = 0, 0
    correct1, total1 = 0, 0

    # recall/precision 계산용
    tp15, fp15, fn15 = 0, 0, 0
    tp1, fp1, fn1 = 0, 0, 0

    for i in range(test_path.shape[0]):
        query = test_path['text_x'].iloc[i]
        target_id = int(test_path['corpus-id'].iloc[i])
        score = int(test_path['score'].iloc[i])   # gold label (1=정답, 0=오답)

        results = search_inference(query, corpus_key, variant, use_gpu_for_faiss)
        idx100 = build_tail100_index(docs, corpus_ids, n=100)

        mapped = attach_corpus_ids_to_results(results, idx100, n=100, on_collision="first")
        mapped_df = pd.DataFrame(mapped, columns=["corpus-id", "text", "score"])
        valid_df = mapped_df.dropna(subset=['corpus-id'])

        if valid_df.empty:
            continue

        answer_list = [int(ans) for ans in valid_df['corpus-id']]
        answer_top1 = int(valid_df.sort_values(by='score', ascending=False)['corpus-id'].iloc[0])

        # === positive sample (score=1) ===
        if score == 1:
            total15 += 1
            total1 += 1
            # acc15
            if target_id in answer_list[:15]:
                correct15 += 1
                tp15 += 1
            else:
                fn15 += 1
            # acc1
            if answer_top1 == target_id:
                correct1 += 1
                tp1 += 1
            else:
                fn1 += 1

        # === negative sample (score=0) ===
        else:
            # 정답이 없어야 하는데 후보에 target_id가 있으면 FP
            if target_id in answer_list[:15]:
                fp15 += 1
            if answer_top1 == target_id:
                fp1 += 1

    acc15 = correct15 / total15 if total15 > 0 else 0.0
    acc1 = correct1 / total1 if total1 > 0 else 0.0

    # recall, precision 계산
    recall15 = tp15 / (tp15 + fn15) if (tp15 + fn15) > 0 else 0.0
    precision15 = tp15 / (tp15 + fp15) if (tp15 + fp15) > 0 else 0.0

    recall1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    precision1 = tp1 / (tp1 + fp1) if (tp1 + fp1) > 0 else 0.0

    print(f"Acc@15: {acc15:.4f} ({correct15}/{total15})")
    print(f"Acc@1:  {acc1:.4f} ({correct1}/{total1})")
    print(f"Recall@15: {recall15:.4f}, Precision@15: {precision15:.4f}")
    print(f"Recall@1:  {recall1:.4f}, Precision@1:  {precision1:.4f}")

    return {
        "acc1": acc1,
        "acc15": acc15,
        "recall1": recall1,
        "precision1": precision1,
        "recall15": recall15,
        "precision15": precision15,
    }



In [17]:
### 학습 전 두 검색기의 성능 측정

In [18]:
## non_train_acc1, non_train_acc15 = evaluate(all_corpus=all_corpus, test_path=test_path, corpus_key="corpus-all", variant="bge-m3@bm25", use_gpu_for_faiss=False)

In [19]:
## non_train_acc1, non_train_acc15 = evaluate(all_corpus=all_corpus, test_path=test_path, corpus_key="corpus-all", variant="ko-legal-sbert@bm25", use_gpu_for_faiss=False)

In [20]:
### 학습 후 두 검색기의 성능 측정

In [1]:
## bge-m3 + bm25 + 증강데이터 x
eval_result = evaluate(all_corpus=all_corpus, test_path=test_path, corpus_key="corpus-train", variant="bge-m3@faiss", use_gpu_for_faiss=False)

NameError: name 'evaluate' is not defined

In [22]:
# bge-m3 + bm25 + 증강데이터 o
eval_result = evaluate(all_corpus=all_corpus, test_path=test_path, corpus_key="corpus-generated-train-50", variant="bge-m3@faiss", use_gpu_for_faiss=False)

Acc@15: 0.0235 (4/170)
Acc@1:  0.0000 (0/170)
Recall@15: 0.0235, Precision@15: 1.0000
Recall@1:  0.0000, Precision@1:  0.0000
